In [1]:
import numpy as np
import pandas as pd
from numpy import kron, trace, sqrt
from functools import reduce
import dill
import sympy as sp
from repeater_helper import merge_state
from scipy.optimize import differential_evolution

q = sp.Symbol("q")
lam = sp.Symbol("lambda")
p_r = sp.Symbol("p_r")
p_l = sp.Symbol("p_l")

H = np.array([[1 / sqrt(2), 1 / sqrt(2)], [1 / sqrt(2), -1 / sqrt(2)]])
H3 = np.kron(np.kron(H, H), H)
def kron_all(*ops):
    return reduce(kron, ops)


def normalize(dm):
    return dm / trace(dm)


identity = np.array([[1, 0], [0, 1]])
x_gate = np.array([[0, 1], [1, 0]])
proj_0 = np.array([[1, 0], [0, 0]])
proj_1 = np.array([[0, 0], [0, 1]])

ket_0 = np.array([[1], [0]])
bra_0 = ket_0.T
ket_1 = np.array([[0], [1]])
bra_1 = ket_1.T

kets = [ket_0, ket_1]
bras = [bra_0, bra_1]

identity = np.array([[1, 0], [0, 1]])
x_gate = np.array([[0, 1], [1, 0]])
y_gate = np.array([[0, -1j], [1j, 0]])
z_gate = np.array([[1, 0], [0, -1]])
pauli_gates = [identity, x_gate, y_gate, z_gate]

proj_0 = np.array([[1, 0], [0, 0]])
proj_1 = np.array([[0, 0], [0, 1]])
ket_0 = np.array([[1], [0]])
ket_1 = np.array([[0], [1]])

U_03 = kron_all(proj_0, identity, identity, identity, identity, identity) + kron_all(
    proj_1, identity, identity, x_gate, identity, identity
)
U_14 = kron_all(identity, proj_0, identity, identity, identity, identity) + kron_all(
    identity, proj_1, identity, identity, x_gate, identity
)
U_25 = kron_all(identity, identity, proj_0, identity, identity, identity) + kron_all(
    identity, identity, proj_1, identity, identity, x_gate
)

ket_abc000 = kron_all(identity, identity, identity, ket_0, ket_0, ket_0)
ket_abc111 = kron_all(identity, identity, identity, ket_1, ket_1, ket_1)


def apply_AD(dm):
    input_dm = kron(dm, dm)

    # rotate
    input_dm = kron(H3, H3) @ input_dm @ kron(H3, H3)
    output_dm = U_03 @ (U_14 @ (U_25 @ input_dm @ U_25.T) @ U_14.T) @ U_03.T
    output_dm = (ket_abc000.T @ output_dm @ ket_abc000) + (
        ket_abc111.T @ output_dm @ ket_abc111
    )
    prob_suc = trace(output_dm)

    # rotate back
    output_dm = H3 @ output_dm @ H3
    return normalize(output_dm), prob_suc
with open("hybrid_prob_click.dill", "rb") as f:
    prob_click = dill.load(f)

with open("hybrid_prob_load.dill", "rb") as f:
    prob_load = dill.load(f)

with open("hybrid_ion_dm.dill", "rb") as f:
    ion_dm = dill.load(f)

get_prob_click = sp.lambdify([q, lam, p_r, p_l], prob_click)
get_prob_load = sp.lambdify([q, lam, p_r, p_l], prob_load)
get_ion_dm = sp.lambdify([q, lam, p_r, p_l], ion_dm)

In [2]:
def get_phase_QBER(dm):
    """
    Extract the phase error rate for the W-state protocol given a density matrix.
    The density matrix should be in the COMPUTATIONAL basis.
    """

    # ZZZ = 1
    # Components p_000, p_011, p_101, p_110
    ZZZ_p = dm[0, 0] + dm[3, 3] + dm[5, 5] + dm[6, 6]

    # ZZZ = -1
    # Components p_111, p_001, p_010, p_100
    ZZZ_m = dm[7, 7] + dm[1, 1] + dm[2, 2] + dm[4, 4]

    ZZZ_exp = ZZZ_p - ZZZ_m

    QBER = (1 + ZZZ_exp) / 2

    return QBER


def get_bit_QBER(dm):
    """
    Extract the bit error rate of the W-state protocol given a density matrix.
    The density matrix should be in the COMPUTATIONAL basis.
    """
    rotated_dm = H3 @ dm @ H3
    # X_1 X_2 = 1 (Alice and Bob are the same)
    # Components p_000, p_001, p_110, p_111

    XX_AB_p = rotated_dm[0, 0] + rotated_dm[1, 1] + rotated_dm[6, 6] + rotated_dm[7, 7]

    # X_1 X_2 = -1 (Alice and Bob are different)
    XX_AB_m = 1 - XX_AB_p

    # X_1 X_3 = 1 (Alice and Charlie are the same)
    # Components p_000, p_101, p_010, p_111

    XX_AC_p = rotated_dm[0, 0] + rotated_dm[5, 5] + rotated_dm[2, 2] + rotated_dm[7, 7]

    # X_1 X_3 = -1 (Alice and Charlie are different)
    XX_AC_m = 1 - XX_AC_p

    XX_AB_exp = XX_AB_p - XX_AB_m
    XX_AC_exp = XX_AC_p - XX_AC_m

    QBER_AB = float((1 - XX_AB_exp) / 2)
    QBER_AC = float((1 - XX_AC_exp) / 2)

    return max(QBER_AB, QBER_AC)


def get_bin_entropy(p):
    """
    Return the binary entropy.
    """
    if p == 0 or p == 1:
        return 0
    else:
        return (-p * (np.log2(p))) + (-(1 - p) * np.log2(1 - p))


def get_loss_ratio(distance_km, loss_db_per_km=0.3):
    """
    Calculate signal loss percentage over a given distance in km,
    based on attenuation in dB/km.

    Parameters:
    - distance_km: float or np.ndarray
    - loss_db_per_km: float, default is 0.3 dB/km

    Returns:
    - loss_percent: float or np.ndarray
    """
    total_loss_db = loss_db_per_km * distance_km
    transmission_ratio = 10 ** (-total_loss_db / 10)
    loss_ratio = 1 - transmission_ratio
    return loss_ratio


def get_sk_fraction(dm):
    bit_QBER = get_bit_QBER(dm)
    phase_QBER = get_phase_QBER(dm)

    return 1 - get_bin_entropy(phase_QBER) - get_bin_entropy(bit_QBER)


def get_key_rate(vars, d, p_l_val, num_level):
    q_val = vars[0]
    lam_val = vars[1]
    t_SPDC = 1
    t_ion = 100
    d_ES = d / (2 ** (num_level + 1))
    t_signal = 2 * d_ES / C
    t_cnot = 100
    t_meas = 100

    p_r_val = get_loss_ratio(d_ES)
    dm_val = get_ion_dm(q_val, lam_val, p_r_val, p_l_val)
    prob_click = 1 - (1 - get_prob_click(q_val, lam_val, p_r_val, p_l_val)) ** NUM_FREQ
    prob_load = get_prob_load(q_val, lam_val, p_r_val, p_l_val)
    prob_EL = prob_click * prob_load

    t_end = (t_SPDC + t_signal + t_ion + t_signal) / prob_EL
    t_merge_ops = t_cnot + t_meas
    t_merge_signal = 2 * t_signal
    t_merge = 2 * (t_merge_ops + t_merge_signal)
    prob_merge = 1

    # Nesting level recursion
    for i in range(num_level):
        dm_val, prob_merge = merge_state(dm_val)
        if i == 0:
            prefactor = 11 / 6
        else:
            prefactor = 3
        t_end = (prefactor * t_end + t_merge) / prob_merge
        t_merge_signal = t_merge_signal * 2
        t_merge = 2 * (t_merge_ops + t_merge_signal)

    t_round = t_end + t_meas
    dm_val, prob_AD = apply_AD(dm_val)
    sk_rate = (1 / 2) * prob_AD * (get_sk_fraction(dm_val) / t_round) * (10**6)
    return sk_rate

C = 0.2
NUM_FREQ = 100
bounds = [(0, 0.5), (0, 0.3)]
p_l_val = 0.01
d_lst = np.linspace(0, 500, 20)

opt_key_rates_level_0 = np.zeros_like(d_lst)
opt_qs_level_0 = np.zeros_like(d_lst)
opt_lambdas_level_0 = np.zeros_like(d_lst)
num_of_level = 0
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_0[i] = -result.fun
    opt_qs_level_0[i] = result.x[0]
    opt_lambdas_level_0[i] = result.x[1]

print("Level 0 calculation finished")

opt_key_rates_level_1 = np.zeros_like(d_lst)
opt_qs_level_1 = np.zeros_like(d_lst)
opt_lambdas_level_1 = np.zeros_like(d_lst)
num_of_level = 1
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_1[i] = -result.fun
    opt_qs_level_1[i] = result.x[0]
    opt_lambdas_level_1[i] = result.x[1]

print("Level 1 calculation finished")

opt_key_rates_level_2 = np.zeros_like(d_lst)
opt_qs_level_2 = np.zeros_like(d_lst)
opt_lambdas_level_2 = np.zeros_like(d_lst)
num_of_level = 2
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_2[i] = -result.fun
    opt_qs_level_2[i] = result.x[0]
    opt_lambdas_level_2[i] = result.x[1]

print("Level 2 calculation finished")


df = pd.DataFrame(
    {
        "distance_km": d_lst,
        "keyrate_bps_level_0": opt_key_rates_level_0,
        "keyrate_bps_level_1": opt_key_rates_level_1,
        "keyrate_bps_level_2": opt_key_rates_level_2,
        "opt_qs_level_0": opt_qs_level_0,
        "opt_qs_level_1": opt_qs_level_1,
        "opt_qs_level_2": opt_qs_level_2,
        "opt_lambdas_level_0": opt_lambdas_level_0,
        "opt_lambdas_level_1": opt_lambdas_level_1,
        "opt_lambdas_level_2": opt_lambdas_level_2,
    }
)

df.to_csv("hybrid_p_l_0_01_AD.csv", index=False, sep=",")
p_l_val = 0.1
d_lst = np.linspace(0, 500, 20)

opt_key_rates_level_0 = np.zeros_like(d_lst)
opt_qs_level_0 = np.zeros_like(d_lst)
opt_lambdas_level_0 = np.zeros_like(d_lst)
num_of_level = 0
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_0[i] = -result.fun
    opt_qs_level_0[i] = result.x[0]
    opt_lambdas_level_0[i] = result.x[1]

print("Level 0 calculation finished")

opt_key_rates_level_1 = np.zeros_like(d_lst)
opt_qs_level_1 = np.zeros_like(d_lst)
opt_lambdas_level_1 = np.zeros_like(d_lst)
num_of_level = 1
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_1[i] = -result.fun
    opt_qs_level_1[i] = result.x[0]
    opt_lambdas_level_1[i] = result.x[1]

print("Level 1 calculation finished")

opt_key_rates_level_2 = np.zeros_like(d_lst)
opt_qs_level_2 = np.zeros_like(d_lst)
opt_lambdas_level_2 = np.zeros_like(d_lst)
num_of_level = 2
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_2[i] = -result.fun
    opt_qs_level_2[i] = result.x[0]
    opt_lambdas_level_2[i] = result.x[1]

print("Level 2 calculation finished")


df = pd.DataFrame(
    {
        "distance_km": d_lst,
        "keyrate_bps_level_0": opt_key_rates_level_0,
        "keyrate_bps_level_1": opt_key_rates_level_1,
        "keyrate_bps_level_2": opt_key_rates_level_2,
        "opt_qs_level_0": opt_qs_level_0,
        "opt_qs_level_1": opt_qs_level_1,
        "opt_qs_level_2": opt_qs_level_2,
        "opt_lambdas_level_0": opt_lambdas_level_0,
        "opt_lambdas_level_1": opt_lambdas_level_1,
        "opt_lambdas_level_2": opt_lambdas_level_2,
    }
)

df.to_csv("hybrid_p_l_0_1_AD.csv", index=False, sep=",")
p_l_val = 0.5
d_lst = np.linspace(0, 500, 20)

opt_key_rates_level_0 = np.zeros_like(d_lst)
opt_qs_level_0 = np.zeros_like(d_lst)
opt_lambdas_level_0 = np.zeros_like(d_lst)
num_of_level = 0
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_0[i] = -result.fun
    opt_qs_level_0[i] = result.x[0]
    opt_lambdas_level_0[i] = result.x[1]

print("Level 0 calculation finished")

opt_key_rates_level_1 = np.zeros_like(d_lst)
opt_qs_level_1 = np.zeros_like(d_lst)
opt_lambdas_level_1 = np.zeros_like(d_lst)
num_of_level = 1
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_1[i] = -result.fun
    opt_qs_level_1[i] = result.x[0]
    opt_lambdas_level_1[i] = result.x[1]

print("Level 1 calculation finished")

opt_key_rates_level_2 = np.zeros_like(d_lst)
opt_qs_level_2 = np.zeros_like(d_lst)
opt_lambdas_level_2 = np.zeros_like(d_lst)
num_of_level = 2
for i in range(len(d_lst)):
    result = differential_evolution(
        lambda v, *args: -get_key_rate(v, *args),
        strategy="best1bin",
        args=(d_lst[i], p_l_val, num_of_level),
        maxiter=1000,
        popsize=20,
        tol=1e-12,
        polish=False,
        bounds=bounds,
        disp=False,
    )

    opt_key_rates_level_2[i] = -result.fun
    opt_qs_level_2[i] = result.x[0]
    opt_lambdas_level_2[i] = result.x[1]

print("Level 2 calculation finished")


df = pd.DataFrame(
    {
        "distance_km": d_lst,
        "keyrate_bps_level_0": opt_key_rates_level_0,
        "keyrate_bps_level_1": opt_key_rates_level_1,
        "keyrate_bps_level_2": opt_key_rates_level_2,
        "opt_qs_level_0": opt_qs_level_0,
        "opt_qs_level_1": opt_qs_level_1,
        "opt_qs_level_2": opt_qs_level_2,
        "opt_lambdas_level_0": opt_lambdas_level_0,
        "opt_lambdas_level_1": opt_lambdas_level_1,
        "opt_lambdas_level_2": opt_lambdas_level_2,
    }
)
df.to_csv("hybrid_p_l_0_5_AD.csv", index=False, sep=",")

Level 0 calculation finished
Level 1 calculation finished
Level 2 calculation finished
Level 0 calculation finished
Level 1 calculation finished
Level 2 calculation finished
Level 0 calculation finished
Level 1 calculation finished
Level 2 calculation finished
